# COCO Segmentation DataLoader Demo
This notebook demonstrates the newly implemented `COCOSegmentationDataset` and how it handles polygon annotations through the refactored inheritance structure.

In [ ]:
import torch
from omegaconf import OmegaConf
from yolo.config.config import DataConfig, DatasetConfig
from yolo.data.loader import create_dataloader
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import numpy as np
from pathlib import Path
import os
from yolo.utils.drawer import draw_masks


## 1. Load Configurations
We use the same `coco.yaml` but initialize it for a segmentation task.

In [ ]:
# Load the dataset config (COCO)
coco_yaml_path = "/home/shrey/projects/yolo/yolo/config/dataset/coco.yaml"
dataset_cfg_dict = OmegaConf.to_container(OmegaConf.load(coco_yaml_path), resolve=True)
dataset_cfg = DatasetConfig(**dataset_cfg_dict)

# Set up the DataConfig
data_cfg = DataConfig(
    shuffle=True,
    batch_size=4,
    pin_memory=False,
    dataloader_workers=0,
    image_size=[640, 640],
    data_augment={},
    source=None,
    dynamic_shape=False
)

print(f"✅ Loading segmentation dataset from: {dataset_cfg.path}")

## 2. Initialize Segmentation DataLoader
Calling `create_dataloader` with `task="segmentation"` will now automatically use `COCOSegmentationDataset`.

In [ ]:
dataloader = create_dataloader(data_cfg, dataset_cfg, task="segment")
print("✅ Segmentation DataLoader created successfully.")

## 3. Fetch and Inspect a Batch
Notice the new `batch.masks` attribute which contains the polygon data.

In [ ]:
batch = next(iter(dataloader))

print(f"Batch object: {type(batch)}")
print(f"Images shape: {batch.images.shape}")
print(f"Targets (Boxes) shape: {batch.targets.shape}")
print(f"Masks type: {type(batch.masks)} (List of polygons per image)")

if batch.masks:
    num_masks = len(batch.masks[0]) if batch.masks[0] is not None else 0
    print(f"Number of masks in first image: {num_masks}")

## 4. Visualization of Polygons
We render the polygons directly onto the image to verify they are loaded and scaled correctly.

In [ ]:
num_to_show = 4
fig, axes = plt.subplots(1, num_to_show, figsize=(20, 6))

for i in range(num_to_show):
    print(batch.images[i].shape)
    img_with_masks = draw_masks(
        batch.images[i], 
        batch.masks[i], 
        idx2label=dataset_cfg.class_list
    )
    
    axes[i].imshow(img_with_masks)
    axes[i].set_title(f"File: {Path(batch.paths[i]).name}", fontsize=10)
    axes[i].axis('off')

plt.tight_layout()
plt.show()
